# VoiceForge -- Day 7: Generate DPO Candidate Pairs

**Goal for today:** for each brief, generate 2 completions from the
**fine-tuned SFT model** (not base) at two different sampling settings,
so Day 8 has real candidate pairs to label chosen/rejected from.

This loads the base model fresh and attaches pushed SFT adapter --
it does not depend on Day 6's live session. Briefs come straight from the
Hub dataset (all three splits combined), so there's no need to re-upload
`raw_prompts.jsonl`.

**Before running:** `Runtime > Change runtime type > T4 GPU`, `HF_TOKEN`
Colab secret set.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub


## 2. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected -- set Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
major, minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Compute capability: {major}.{minor} -> using {'bf16' if USE_BF16 else 'fp16'}")


## 3. Mount Drive, log in, load briefs from the Hub dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/voiceforge"
Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

from datasets import load_dataset, concatenate_datasets

SFT_DATASET_REPO = "bikalpoudel/voiceforge-brand-voice-sft"
ds = load_dataset(SFT_DATASET_REPO)

# reuse the same briefs SFT was trained on, across all three splits --
# this is what Day 7 of the plan calls for. We only need the brief/voice
# metadata here, not the original SFT completion.
all_rows = concatenate_datasets([ds["train"], ds["validation"], ds["test"]])
print(f"{len(all_rows)} briefs available for DPO candidate generation")

DPO_CANDIDATE_N = None  # None = use all; set an int (e.g. 150) to cap for a faster/cheaper first pass
briefs = all_rows.shuffle(seed=42)
if DPO_CANDIDATE_N:
    briefs = briefs.select(range(min(DPO_CANDIDATE_N, len(briefs))))
print(f"Generating candidates for {len(briefs)} briefs")


## 4. Load base model + attach the pushed SFT adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
SFT_ADAPTER_REPO = "bikalpoudel/voiceforge-brand-voice-sft-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
)

model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_REPO)
model.eval()
model.config.use_cache = True  # inference, not training -- cache speeds up generation

print(f"Loaded {BASE_MODEL} + SFT adapter {SFT_ADAPTER_REPO}")
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. System prompts (must match what SFT trained on)

Generation has to use the same system prompt phrasing the model was
fine-tuned against -- these are copied verbatim from the Day 6 notebook's
`DISTILLED_SYSTEM_PROMPTS`. If you trained with
`USE_DISTILLED_SYSTEM_PROMPT = False` in Day 6, swap this cell to pull
`messages[0]["content"]` from the dataset rows instead, or your DPO
candidates will look worse than the model actually is, for reasons
that have nothing to do with voice quality.


In [ ]:
DISTILLED_SYSTEM_PROMPTS = {
    "cozy_crochet": (
        "You are a copywriter for a cozy handmade crochet shop. Voice: genuine, "
        "cozy, cute, simple, thoughtful. Short sentences, contractions and "
        "fragments are fine. Emoji: choose from \U0001F9F6 \U0001F338 \u2728 \U0001F49B, max 1-2 per post. Never say: "
        "elevate, premium, luxurious, game-changer, exclusive drop. Include one "
        "concrete detail (size, turnaround time, or material) when the format "
        "calls for it. Output only the copy itself, no labels or quotation marks."
    ),
    "romantic_floral": (
        "You are a copywriter for a romantic handmade bouquet shop. Voice: "
        "sentimental, heartfelt, aesthetic, warm, custom-focused. Emoji: choose "
        "from \U0001F490 \U0001F380 \U0001F48C \u2728, max 1-2 per post. Never say: unlock, revolutionary, "
        "unrivaled, cheap, bulk, flash sale. Include one concrete detail (size, "
        "turnaround time, or materials) when the format calls for it. Output "
        "only the copy itself, no labels or quotation marks."
    ),
}


## 6. Generate 2 candidates per brief

Candidate A: conservative sampling (temperature 0.6) -- tends to stay
close to what SFT learned, usually more reliably on-voice.
Candidate B: loose sampling (temperature 1.2, top_p 0.98) -- more likely
to drift off-voice, ramble past the length target, or drop the CTA. This
"vary the sampling" approach is what actually produces a genuine
contrast for DPO to learn from, rather than two near-identical
completions that don't teach the model anything when paired up.

Resumable: safe to stop and rerun, already-generated ids are skipped.


In [ ]:
import json
import time

OUT_PATH = f"{DRIVE_PROJECT_DIR}/dpo_candidates.jsonl"

CANDIDATE_A_PARAMS = dict(do_sample=True, temperature=0.6, top_p=0.9)
CANDIDATE_B_PARAMS = dict(do_sample=True, temperature=1.2, top_p=0.98)

def generate(brief, voice, gen_kwargs, max_new_tokens=120):
    messages = [
        {"role": "system", "content": DISTILLED_SYSTEM_PROMPTS[voice]},
        {"role": "user", "content": brief},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            **gen_kwargs,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def already_done_ids(path):
    if not Path(path).exists():
        return set()
    return {json.loads(l)["id"] for l in Path(path).read_text().splitlines() if l.strip()}

done = already_done_ids(OUT_PATH)
print(f"{len(done)} of {len(briefs)} already generated. Resuming...")

with open(OUT_PATH, "a", encoding="utf-8") as f:
    for i, row in enumerate(briefs):
        if row["id"] in done:
            continue
        candidate_a = generate(row["brief"], row["voice"], CANDIDATE_A_PARAMS)
        candidate_b = generate(row["brief"], row["voice"], CANDIDATE_B_PARAMS)

        out_row = {
            "id": row["id"],
            "voice": row["voice"],
            "voice_tag": row["voice_tag"],
            "format": row["format"],
            "product": row["product"],
            "angle": row["angle"],
            "brief": row["brief"],
            "candidate_a": candidate_a,
            "candidate_a_params": CANDIDATE_A_PARAMS,
            "candidate_b": candidate_b,
            "candidate_b_params": CANDIDATE_B_PARAMS,
        }
        f.write(json.dumps(out_row) + "\n")
        f.flush()

        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(briefs)} done")

print(f"Done. Output written to {OUT_PATH}")


## 7. Quick diversity check

A candidate pair that's nearly identical text doesn't teach DPO
anything when it gets labeled chosen/rejected later -- it's a wasted
example. This flags pairs that are suspiciously similar so Day 8 can
prioritize reviewing (or dropping) them, using the same character-level
similarity idea, not a hard filter at this stage.


In [ ]:
import difflib
from collections import Counter

rows = [json.loads(l) for l in open(OUT_PATH)]

def similarity(a, b):
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()

sims = [similarity(r["candidate_a"], r["candidate_b"]) for r in rows]
near_identical = sum(1 for s in sims if s > 0.85)

print(f"{len(rows)} candidate pairs generated")
print(f"Average A/B similarity: {sum(sims) / len(sims):.2f}")
print(f"Near-identical pairs (similarity > 0.85): {near_identical} "
      f"({near_identical / len(rows):.0%}) -- review these closely in Day 8, "
      f"weak contrast makes for a weak DPO example")

print()
print("--- Example pair ---")
example = rows[0]
print("VOICE:", example["voice"], "| FORMAT:", example["format"])
print("BRIEF:", example["brief"])
print("CANDIDATE A (temp 0.6):", example["candidate_a"])
print("CANDIDATE B (temp 1.2):", example["candidate_b"])


## What to check before Day 8

- `data/dpo_candidates.jsonl` (on Drive) has one row per brief with two
  distinct candidate completions.
- Skim a handful of pairs yourself -- do A and B actually read
  differently in tone/quality, or does the model just produce the same
  good output regardless of temperature (a sign the SFT model is very
  confident, which is good for SFT quality but means DPO may need a
  larger temperature gap or a different contrast strategy)?
- Note the near-identical percentage from the diversity check -- if it's
  high (>30-40%), consider widening the gap between Candidate A/B
  sampling settings before moving on, rather than labeling weak pairs.

Day 8 takes this file and produces `chosen`/`rejected` labels per pair,
scored against `voice_guidelines.md`'s rubric.
